In [0]:
spark.conf.set(
  "fs.azure.account.key.adlsvenproject1.dfs.core.windows.net",
  dbutils.secrets.get(scope="secret-scope-azure-live-project-1", key="storage-key")
)

df_header = spark.read.parquet(
    "abfss://datalake-ven-project1@adlsvenproject1.dfs.core.windows.net/bronze/sales_order_header/"
)

#display(df_header)

df_detail = spark.read.parquet(
    "abfss://datalake-ven-project1@adlsvenproject1.dfs.core.windows.net/bronze/sales_order_detail/"
)

#display(df_detail)

#df_product = spark.read.parquet(
#    "abfss://datalake-ven-project1@adlsvenproject1.dfs.core.windows.net/bronze/product/"
#)

#display(df_product)


In [0]:
from pyspark.sql.functions import col

df_h = df_header.select(
    col("SalesOrderID"),
    col("OrderDate"),
    col("CustomerID")
)

df_d = df_detail.select(
    col("SalesOrderID"),
    col("SalesOrderDetailID"),
    col("ProductID"),
    col("OrderQty"),
    col("LineTotal"),
    col("ModifiedDate")  # IMPORTANT for dedup
)

df_p = df_product.select(
    col("ProductID"),
    col("Name").alias("ProductName")
)

In [0]:
df_join = df_h.join(df_d, "SalesOrderID", "inner") \
              .join(df_p, "ProductID", "left")

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window_spec = Window.partitionBy("SalesOrderDetailID") \
                    .orderBy(col("ModifiedDate").desc())

df_dedup = df_join.withColumn("rn", row_number().over(window_spec)) \
                  .filter(col("rn") == 1) \
                  .drop("rn")

In [0]:
df_silver = df_dedup.select(
    "SalesOrderDetailID",
    "SalesOrderID",
    "ProductID",
    "ProductName",
    "CustomerID",
    "OrderQty",
    "LineTotal",
    "ModifiedDate"   # ✅ MUST include
)

In [0]:
#dbutils.fs.rm(silver_path, True)

In [0]:
silver_path = "abfss://datalake-ven-project1@adlsvenproject1.dfs.core.windows.net/silver/sales_data/"
#df_silver.write.format("delta").mode("overwrite").save(silver_path)

In [0]:
df_silver.count()

In [0]:
from pyspark.sql.functions import count

df_silver.groupBy("SalesOrderDetailID") \
         .count() \
         .filter("count > 1") \
         .display()

In [0]:
merge_result = delta_table.alias("t").merge(
    df_silver.alias("s"),
    "t.SalesOrderDetailID = s.SalesOrderDetailID"
).whenMatchedUpdate(
    condition="t.ModifiedDate < s.ModifiedDate",
    set={
        "SalesOrderID": "s.SalesOrderID",
        "ProductID": "s.ProductID",
        "CustomerID": "s.CustomerID",
        "OrderQty": "s.OrderQty",
        "LineTotal": "s.LineTotal",
        "ModifiedDate": "s.ModifiedDate"
    }
).whenNotMatchedInsertAll() \
.execute()

In [0]:
display(merge_result)